# Machine Translation using Encoder-Decoder

In [ ]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 132.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
from torch import nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
from nltk.tokenize import word_tokenize
import spacy
import re
import string
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
import kagglehub
path = kagglehub.dataset_download("shahadhamza/multi30k-dataset")
print(path)

Using Colab cache for faster access to the 'multi30k-dataset' dataset.
/kaggle/input/multi30k-dataset


In [ ]:
en_file_path = "/kaggle/input/multi30k-dataset/train.en"
de_file_path = "/kaggle/input/multi30k-dataset/train.fr"
SOS_token = 0
EOS_token = 1
device = 'cuda' if torch.cuda.is_available() else "cpu"
spacy_en = spacy.load("en_core_web_sm")
spacy_fr = spacy.load("fr_core_news_sm")

In [ ]:
english_file_instance = open('/kaggle/input/multi30k-dataset/train.en','r')
france_file_instance = open('/kaggle/input/multi30k-dataset/train.fr','r')
english_data = english_file_instance.readlines()
france_data = france_file_instance.readlines()

In [ ]:
print("data summary")
print("----------------------------------------------------")
print('English Data')
print(f"number of samples : {len(english_data)}")
print(f"example : {english_data[0]}")
print("----------------------------------------------------")
print("french Data")
print(f"number of samples : {len(france_data)}")
print(f"example : {france_data[0]}")

data summary
----------------------------------------------------
English Data
number of samples : 29000
example : Two young, White males are outside near many bushes.

----------------------------------------------------
french Data
number of samples : 29000
example : Deux jeunes hommes blancs sont dehors près de buissons.



**Preprocessing Data**

In [ ]:
def preprocess_tokens(tokens):
    cleaned_tokens = []
    for token in tokens:
        token = token.lower()
        token = token.strip()
        if re.search(r'\d', token):
            continue
        if token in string.punctuation:
            continue
        if not token:
            continue
        cleaned_tokens.append(token)
    return cleaned_tokens


def tokenize_eng(text):
  return [tok.text.lower() for tok in spacy_en.tokenizer(text)]

def tokenize_fr(text):
  return [tok.text.lower() for tok in spacy_fr.tokenizer(text)]

def get_cleaned_tokens(sentence, lang='en'):
  if lang == 'en':
      raw_tokens = tokenize_eng(sentence)
  else:
      raw_tokens = tokenize_fr(sentence)
  return preprocess_tokens(raw_tokens)


def sentence_tokenizer(sentence, vocab, lang='en'):
    tokens = tokenize_eng(sentence) if lang == 'en' else tokenize_fr(sentence)
    indices = [vocab.get(token, 0) for token in tokens]
    indices = [vocab["<sos>"]] + indices + [vocab["<eos>"]]
    return indices

In [ ]:
en_vocab = set()
fr_vocab = set()

en_vocab = {"<unk>":0,"<pad>":1,"<sos>":2,"<eos>":3}
fr_vocab = {"<unk>":0,"<pad>":1,"<sos>":2,"<eos>":3}

def build_vocab(sentence,lang='en'):
  vocab = en_vocab if lang == 'en' else fr_vocab
  tokens = get_cleaned_tokens(sentence,lang)
  for token in tokens:
    if token not in vocab:
      vocab[token] = len(vocab)


tokenized_en_sentences = []
tokenized_fr_sentences = []

# building Vocabs
for sentence in english_data:
  build_vocab(sentence,'en')
for sentence in france_data:
  build_vocab(sentence,'fr')

en_index_to_word = {i: w for w, i in en_vocab.items()}
fr_index_to_word = {i: w for w, i in fr_vocab.items()}

# and tokenize sentence
for sentence in english_data:
  tokenized_en_sentences.append(sentence_tokenizer(sentence,en_vocab,'en'))
for sentence in france_data:
  tokenized_fr_sentences.append(sentence_tokenizer(sentence,fr_vocab,'fr'))

In [ ]:
print("Vocabulary summary : ")
print("---------------------------------------------")
print("English Vocab : ")
print(f"number of words : {len(en_vocab)}")
print("sample from english vocab : ")
for key,value in list(en_vocab.items())[5:10]:
  print(key,":",value)
print("---------------------------------------------")
print("France Vocab : ")
print(f"number of words : {len(fr_vocab)}")
print("sample from french vocab : ")
for key,value in list(fr_vocab.items())[5:10]:
  print(key,":",value)

Vocabulary summary : 
---------------------------------------------
English Vocab : 
number of words : 9701
sample from english vocab : 
young : 5
white : 6
males : 7
are : 8
outside : 9
---------------------------------------------
France Vocab : 
number of words : 11070
sample from french vocab : 
jeunes : 5
hommes : 6
blancs : 7
sont : 8
dehors : 9


In [42]:
print("Tokenized Sentence summary : ")
print("---------------------------------------------")
print("English Sentence : ")
print(f"number of Sentence : {len(tokenized_en_sentences)}")
print("sample from english Sentence : ")
print(tokenized_en_sentences[9])
print("Human readable form : ")
sentence = ""
for token in tokenized_en_sentences[9]:
  sentence = sentence +" "+ en_index_to_word[token]
print(sentence)
print("---------------------------------------------")
print("France Sentence : ")
print(f"number of Sentence : {len(tokenized_fr_sentences)}")
print("sample from french Sentence : ")
print(tokenized_fr_sentences[9])
print("Human readable form : ")
sentence = ""
for token in tokenized_fr_sentences[9]:
  sentence = sentence +" "+ fr_index_to_word[token]
print(sentence)


Tokenized Sentence summary : 
---------------------------------------------
English Sentence : 
number of Sentence : 29000
sample from english Sentence : 
[2, 68, 69, 34, 70, 15, 39, 71, 72, 39, 73, 0, 0, 3]
Human readable form : 
 <sos> boys dancing on poles in the middle of the night <unk> <unk> <eos>
---------------------------------------------
France Sentence : 
number of Sentence : 29000
sample from french Sentence : 
[2, 70, 71, 72, 34, 70, 73, 74, 75, 11, 61, 76, 0, 0, 3]
Human readable form : 
 <sos> des garçons dansent sur des barres au milieu de la nuit <unk> <unk> <eos>


In [ ]:
class Eng_Fre_dataset(Dataset):
  def __init__(self,en_data,fr_data):
    self.en_data = [torch.tensor(s) for s in en_data]
    self.fr_data = [torch.tensor(s) for s in fr_data]

  def __len__(self):
    return len(self.en_data)

  def __getitem__(self,idx):
    return self.en_data[idx],self.fr_data[idx]

def collate_fn(batch):
  en_batch, fr_batch = zip(*batch)
  en_padded = pad_sequence(sequences = en_batch , batch_first = True, padding_value=1)
  fr_padded = pad_sequence(sequences = fr_batch , batch_first = True, padding_value=1)
  return en_padded , fr_padded

In [ ]:
dataset = Eng_Fre_dataset(tokenized_en_sentences,tokenized_fr_sentences)
loader = DataLoader(
    dataset = dataset,
    batch_size = 32,
    collate_fn = collate_fn,
    shuffle=True,
)

In [ ]:
class Encoder(nn.Module):
  def __init__(self,vocab_size,embed_size,hidden_size,n_layers,dropout):
    super().__init__()
    self.embedding_layer = nn.Embedding(vocab_size,embed_size)
    self.rnn_layer = nn.LSTM(embed_size,hidden_size,n_layers,batch_first=True,dropout=dropout)
    self.dropout = nn.Dropout(dropout)

  def forward(self,input):
    x = self.dropout(self.embedding_layer(input))
    output , (hidden,cell) = self.rnn_layer(x)
    return hidden,cell


In [ ]:
class Decoder(nn.Module):
  def __init__(self,vocab_size,emmeb_size,hidden_size,n_layers,dropout):
    super().__init__()
    self.output_dim = vocab_size
    self.embedding_layer = nn.Embedding(vocab_size,emmeb_size)
    self.rnn_layer = nn.LSTM(emmeb_size,hidden_size,num_layers=n_layers,dropout=dropout,batch_first=True)
    self.fc_out = nn.Linear(hidden_size,vocab_size)
    self.dropout_layer = nn.Dropout(dropout)

  def forward(self,input,hidden,cell):
    input = input.unsqueeze(1)
    embedded = self.dropout_layer(self.embedding_layer(input))
    output, (hidden,cell) = self.rnn_layer(embedded,(hidden,cell))
    prediction = self.fc_out(output.squeeze(1))

    return prediction,hidden,cell


In [ ]:
import random

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5): # src : input for encoder, trg : input for decoder
        batch_size = trg.shape[0] #32
        trg_len = trg.shape[1] # seq_len
        trg_vocab_size = self.decoder.output_dim # 29000

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)
        input = trg[:, 0] # 1 word at a time

        # process one word at a particular timestep t
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            outputs[:, t] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:, t] if teacher_force else top1
        return outputs

In [ ]:
en_vocab_size = len(en_vocab)
fr_vocab_size = len(fr_vocab)

encoder = Encoder(en_vocab_size,256,512,2,0.5)
decoder = Decoder(fr_vocab_size,256,512,2,0.5)

model = Seq2Seq(encoder,decoder,device).to(device)

epochs = 50
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
loss_function = nn.CrossEntropyLoss(ignore_index=1)

CLIP = 1
best_valid_loss = float('inf')


In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for i, (src, trg) in enumerate(iterator):
        src = src.to(device)
        trg = trg.to(device)

        optimizer.zero_grad()
        output = model(src, trg)
        output_dim = output.shape[-1]
        output = output[:, 1:].reshape(-1, output_dim)
        trg = trg[:, 1:].reshape(-1)

        loss = criterion(output, trg)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

In [ ]:
for epoch in range(epochs):
    train_loss = train(model, loader, optimizer, loss_function, CLIP)

    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f}')

    if train_loss < best_valid_loss:
        best_valid_loss = train_loss
        torch.save(model.state_dict(), 'tut1-model.pt')

Epoch: 01 | Train Loss: 4.488
Epoch: 02 | Train Loss: 3.724
Epoch: 03 | Train Loss: 3.400
Epoch: 04 | Train Loss: 3.166
Epoch: 05 | Train Loss: 3.001
Epoch: 06 | Train Loss: 2.833
Epoch: 07 | Train Loss: 2.694
Epoch: 08 | Train Loss: 2.573
Epoch: 09 | Train Loss: 2.464
Epoch: 10 | Train Loss: 2.365
Epoch: 11 | Train Loss: 2.286
Epoch: 12 | Train Loss: 2.196
Epoch: 13 | Train Loss: 2.127
Epoch: 14 | Train Loss: 2.053
Epoch: 15 | Train Loss: 1.999
Epoch: 16 | Train Loss: 1.929
Epoch: 17 | Train Loss: 1.872
Epoch: 18 | Train Loss: 1.820
Epoch: 19 | Train Loss: 1.769
Epoch: 20 | Train Loss: 1.723
Epoch: 21 | Train Loss: 1.683
Epoch: 22 | Train Loss: 1.655
Epoch: 23 | Train Loss: 1.609
Epoch: 24 | Train Loss: 1.578
Epoch: 25 | Train Loss: 1.536
Epoch: 26 | Train Loss: 1.495
Epoch: 27 | Train Loss: 1.466
Epoch: 28 | Train Loss: 1.440
Epoch: 29 | Train Loss: 1.406
Epoch: 30 | Train Loss: 1.388
Epoch: 31 | Train Loss: 1.356
Epoch: 32 | Train Loss: 1.335
Epoch: 33 | Train Loss: 1.316
Epoch: 34 

In [24]:
def translate_sentence(model, sentence, en_vocab, fr_vocab, device, max_len=50):
    model.eval()
    tokens = sentence_tokenizer(sentence, en_vocab)
    src_tensor = torch.LongTensor(tokens).unsqueeze(0).to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(src_tensor)

    trg_indices = [fr_vocab["<sos>"]]
    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indices[-1]]).to(device)
        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)
        pred_token = output.argmax(1).item()
        trg_indices.append(pred_token)
        if pred_token == fr_vocab["<eos>"]:
            break
    translated_tokens = [fr_index_to_word[i] for i in trg_indices]
    return translated_tokens[1:-1]



In [45]:
sentence = input("Enter the sentence to be translated : ")
output = translate_sentence(model,sentence,en_vocab,fr_vocab,device)
print(f"translated output : {" ".join(output)}")

Enter the sentence to be translated : the boys are sitting by the bank of a river
translated output : les garçons sont assis côte à côte d' une rivière <unk> <unk>
